# Prototyp Fazy 3: kontekstowa analiza LLM (spec SPEC.md §6.1 warstwa 3, §7.1, otwarte pytanie #4)

Cel: sprawdzić **realnymi danymi**, czy model językowy wykrywa ataki, które
nasz obecny pipeline (`rules.py` + heurystyka/RandomForest z `ml/train.py`)
**przegapia** -- konkretnie payloady z `EVASIVE_PAYLOADS` w
[ml/dataset.py](../ml/dataset.py), celowo skonstruowane tak, żeby ominąć regex.

**Trzy warianty do wyboru jedną zmienną (`MODEL_SIZE`) w komórce z modelem
-- to jest wprost hosting lokalny vs zewnętrzny z otwartego pytania #4:**
- `"small"` -- Phi-3-mini-4k-instruct (3.8B, 4-bit), lokalnie na darmowym T4
- `"big"` -- Qwen2.5-32B-Instruct (32B, 4-bit), lokalnie na A100, płatne jednostki Colaba
- `"gemini"` -- Gemini przez Google AI Studio API (Twój plan AI Pro) -- **hosting zewnętrzny**,
  zero GPU, zero Colaba nawet -- to samo odpala się na zwykłym laptopie

Ta sama metodyka, ten sam zestaw testowy, ten sam kod liczący metryki dla
wszystkich trzech -- więc wyniki są wprost porównywalne (to samo `df` na
końcu, tylko inny `MODEL_SIZE`).

To jest test, nie wdrożenie -- wynik ma pomóc odpowiedzieć na otwarte pytanie
#4 specyfikacji (hosting lokalny czy zewnętrzny, jaki rozmiar modelu) *danymi*,
zamiast zgadywaniem.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK (OK jesli MODEL_SIZE='gemini' -- to nie potrzebuje GPU)")
!pip install --quiet transformers accelerate bitsandbytes google-genai

In [ ]:
import os, sys

REPO_DIR = "/content/ai-waf-spec"

# Repo publiczne -- zero tokena potrzebne. Idempotentne: bezpieczne do
# wielokrotnego uruchomienia (klonuje tylko jesli jeszcze nie istnieje,
# nigdy nie usuwa katalogu w ktorym akurat stoimy -- to byl zrodlem
# bledu "Unable to read current working directory" przy powtornych probach).
if not os.path.isdir(os.path.join(REPO_DIR, "app")):
    os.chdir("/content")
    !git clone https://github.com/pamsmediatech-lang/ai-waf-spec.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("repo gotowe:", os.path.exists(os.path.join(REPO_DIR, "app", "waf", "request.py")))

In [ ]:
# MODEL_SIZE:
#  "small"  -- Phi-3-mini-4k-instruct, lokalnie, T4 (darmowe)
#  "big"    -- Qwen2.5-32B-Instruct, lokalnie, A100 (platne jednostki)
#  "gemini" -- Gemini przez Google AI Studio API -- hosting zewnetrzny,
#              zero GPU potrzebne, dziala nawet na slabym laptopie
MODEL_SIZE = "gemini"  # "small" | "big" | "gemini"

if MODEL_SIZE == "gemini":
    import getpass
    from google import genai

    GEMINI_API_KEY = getpass.getpass("Google AI Studio API key (aistudio.google.com/apikey): ")
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    # Historia probek (na zywym koncie, wrzesien 2026):
    #  - gemini-3.8-flash: 503 "high demand" (swiezo wypuszczony, przeciazony)
    #  - gemini-2.5-flash: 404 "no longer available" (wycofany z API)
    # gemini-3.5-flash-lite: aktualny, lekki, zoptymalizowany pod wysoki
    # przeplyw zapytan -- mniejsze ryzyko przeciazenia niz flagowy 3.8.
    MODEL_NAME = "gemini-3.5-flash-lite"
    print(f"Gemini API skonfigurowane: {MODEL_NAME} -- hosting zewnetrzny, bez lokalnego GPU")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_CONFIGS = {
        "small": "microsoft/Phi-3-mini-4k-instruct",   # 3.8B -- T4 16GB, wystarcza z zapasem
        "big": "Qwen/Qwen2.5-32B-Instruct",             # 32B -- wymaga A100 (4-bit: ~20GB VRAM)
    }
    MODEL_NAME = MODEL_CONFIGS[MODEL_SIZE]
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
    print(f"model zaladowany lokalnie: {MODEL_NAME} (MODEL_SIZE={MODEL_SIZE!r})")

In [ ]:
import time

from google.genai import types

PROMPT_TEMPLATE = """Jestes analitykiem bezpieczenstwa WAF. Ocen ponizsza wartosc
parametru HTTP (query/body/path) pod katem tego, czy jest to proba ataku
(SQL injection, XSS, path traversal, command injection) czy normalna,
legalna wartosc.

Wartosc: {payload!r}

Odpowiedz DOKLADNIE w formacie:
WERDYKT: ATAK albo BEZPIECZNE
UZASADNIENIE: <jedno zdanie>
"""

# temperature=0: determinizm -- bez tego 3 identyczne przebiegi moga byc
# zbieznoscia losowego samplingu, nie dowodem stabilnosci klasyfikacji.
GEMINI_CONFIG = types.GenerateContentConfig(temperature=0)


def _parse_verdict(text: str) -> str:
    return "ATAK" if "WERDYKT: ATAK" in text.upper().replace(":ATAK", ": ATAK") else "BEZPIECZNE"


def _call_gemini_with_retry(prompt: str, max_attempts: int = 4) -> str:
    # Trzy rozne bledy, trzy rozne strategie (wszystkie napotkane na
    # zywo w tej sesji):
    #  - 503 UNAVAILABLE ("high demand") -- chwilowe przeciazenie
    #    serwera, krotki backoff (5s, 10s, 20s) zwykle wystarcza.
    #  - 429 RESOURCE_EXHAUSTED -- limit zapytan wyczerpany, krotki
    #    backoff nic nie da -- dluzsze oczekiwanie (30s) i jasny komunikat.
    #  - 404 NOT_FOUND ("model no longer available") -- model zostal
    #    wycofany z API (spotkalo to gemini-2.5-flash). Retry tu nie ma
    #    sensu w ogole -- od razu czytelny blad zamiast 4 prob w mur.
    for attempt in range(max_attempts):
        try:
            response = gemini_client.models.generate_content(
                model=MODEL_NAME, contents=prompt, config=GEMINI_CONFIG
            )
            return response.text
        except Exception as exc:
            exc_str = str(exc)
            if "NOT_FOUND" in exc_str or "404" in exc_str:
                raise RuntimeError(
                    f"Model {MODEL_NAME!r} nie istnieje/zostal wycofany z API. "
                    "Sprawdz aktualna liste na ai.google.dev/gemini-api/docs/models "
                    "i zmien MODEL_NAME w komorce wyzej."
                ) from exc
            is_quota = "RESOURCE_EXHAUSTED" in exc_str or "429" in exc_str
            if is_quota and attempt >= 1:
                raise RuntimeError(
                    "Limit zapytan (429 RESOURCE_EXHAUSTED) wyczerpany na koncie/planie. "
                    "Krotkie odczekanie tu nie pomoze -- sprawdz limity na "
                    "aistudio.google.com/apikey (Usage/Billing) albo sprobuj innego modelu."
                ) from exc
            if attempt == max_attempts - 1:
                raise
            wait_s = 30 if is_quota else 5 * (2 ** attempt)
            print(f"  (Gemini chwilowo niedostepne: {exc} -- ponawiam za {wait_s}s)")
            time.sleep(wait_s)


def classify_with_llm(payload: str) -> tuple[str, str]:
    prompt = PROMPT_TEMPLATE.format(payload=payload)

    if MODEL_SIZE == "gemini":
        text = _call_gemini_with_retry(prompt)
    else:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
        output = model.generate(inputs, max_new_tokens=80, do_sample=False, temperature=None, top_p=None)
        text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

    return _parse_verdict(text), text.strip()

In [ ]:
# Zestaw testowy: dokladnie te przypadki, ktore w tests/test_ml_dataset.py
# sluza do sprawdzenia, ze zbior NIE jest trywialny -- payloady, ktore
# rules.py (regex) i heurystyka/RandomForest maja szanse przegapic
# (rule_hit_count == 0), plus kilka jednoznacznie benignych zdan do
# sprawdzenia false-positive rate LLM-a.
#
# Rozmiar zbioru (2026-09-25, po krytyce wiarygodnosci pierwszego
# przebiegu): evasive 5->11, benign 9->20, obvious 5->8. Wciaz nie jest
# to proba statystycznie duza -- patrz SPEC.md par.15 pytanie #4 i
# issue #16 -- ale przedzial ufnosci dla np. 11/11 trafien jest wyraznie
# wezszy niz dla 5/5.
from app.waf.request import WafRequest
from app.waf.rules import evaluate
from ml.dataset import BENIGN_BODIES, EVASIVE_PAYLOADS, MALICIOUS_PAYLOADS

test_cases = []
for category, payload in EVASIVE_PAYLOADS:
    test_cases.append((payload, 1, category, "evasive"))
for category, payload in MALICIOUS_PAYLOADS[:8]:
    test_cases.append((payload, 1, category, "obvious"))
for benign in BENIGN_BODIES:
    if benign:
        test_cases.append((benign, 0, "benign", "benign"))

print(f"{len(test_cases)} przypadkow testowych")

In [ ]:
import time

results = []
for i, (payload, true_label, category, kind) in enumerate(test_cases):
    req = WafRequest(method="GET", path="/", client_ip="0.0.0.0", query={"x": [payload]})
    rule_hit = len(evaluate(req)) > 0
    try:
        verdict, reasoning = classify_with_llm(payload)
        llm_correct = (verdict == "ATAK") == bool(true_label)
    except Exception as exc:
        # Nie wywalaj calej petli na jednym zepsutym wywolaniu -- zapisz
        # jako blad (z PELNA trescia wyjatku w kolumnie "reasoning", nie
        # tylko w printcie ponizej, zeby dalo sie to zdiagnozowac z CSV
        # a nie tylko ze zrzutu ekranu output-u tej komorki) i jedz dalej.
        verdict, reasoning, llm_correct = "BLAD", str(exc)[:300], None
        print(f"  !! blad na payloadzie {payload[:40]!r}: {exc}")
    results.append({
        "model_size": MODEL_SIZE, "model_name": MODEL_NAME,
        "payload": payload[:50], "kind": kind, "category": category,
        "true_label": "ATAK" if true_label else "BEZPIECZNE",
        "rules_caught_it": rule_hit, "llm_verdict": verdict,
        "llm_correct": llm_correct, "reasoning": reasoning,
    })
    print(f"[{kind:8s}] rules={rule_hit!s:5s} llm={verdict:10s} prawda={results[-1]['true_label']:10s} :: {payload[:60]}")
    if MODEL_SIZE == "gemini" and i < len(test_cases) - 1:
        time.sleep(2)  # odstep miedzy wywolaniami -- zapobiega wpadnieciu w limit RPM zamiast go leczyc

n_errors = sum(1 for r in results if r["llm_verdict"] == "BLAD")
print(f"\nGotowe: {len(results)} przypadkow, w tym {n_errors} bledow API (patrz kolumna llm_verdict='BLAD').")

In [ ]:
import pandas as pd

df_all = pd.DataFrame(results)
df = df_all[df_all["llm_verdict"] != "BLAD"].copy()  # metryki tylko na udanych wywolaniach
n_errors = len(df_all) - len(df)

print(f"=== Model: {MODEL_NAME} (MODEL_SIZE={MODEL_SIZE!r}) ===\n")
if n_errors:
    print(f"UWAGA: {n_errors}/{len(df_all)} przypadkow zakonczylo sie bledem API (pominiete w metrykach ponizej).\n")

print("=== Ogolna skutecznosc LLM ===")
print(df.groupby("kind")["llm_correct"].mean())

print("\n=== To jest sedno testu: ewazje, ktore regex przegapil (rules_caught_it=False), ")
print("    czy LLM je mimo to zlapal? ===")
evasive_missed_by_rules = df[(df["kind"] == "evasive") & (~df["rules_caught_it"])]
print(evasive_missed_by_rules[["payload", "llm_verdict", "llm_correct"]])
print(f"\nLLM recall na ewazjach ominietych przez regex: "
      f"{evasive_missed_by_rules['llm_correct'].mean():.1%}")

print("\n=== False positive rate LLM na benignych zdaniach ===")
benign_rows = df[df["kind"] == "benign"]
print(f"FP rate: {(1 - benign_rows['llm_correct']).mean():.1%}")

# Zapisz PELNY wynik (z bledami, jesli byly) do CSV z sufiksem MODEL_SIZE
# -- uruchom notebook dwa razy (raz "small", raz "big"/"gemini") i
# porownaj oba pliki, zeby miec twarda odpowiedz na otwarte pytanie #4,
# a nie wrazenie z jednego przebiegu.
df_all.to_csv(f"/content/ai-waf-spec/notebooks/llm_results_{MODEL_SIZE}.csv", index=False)
print(f"\nZapisano: notebooks/llm_results_{MODEL_SIZE}.csv")

## Jak czytac wynik

- **Wysoki recall na ewazjach + niski FP rate na benignych** -> LLM realnie
  domyka luke z §7.5 (odpornosc na obejscia), warto wdrazac Faze 3 wg planu
  z SPEC.md §10.2 (shadow mode przed wplywem na ruch).
- **Wysoki FP rate** -> model za bardzo "wpada w panike" na zdania
  zawierajace slowa kluczowe SQL w naturalnym jezyku -- potrzeba lepszego
  promptu albo mocniejszego modelu.
- **Niski recall na ewazjach** -> ten model nie daje realnej przewagi nad
  obecnym pipeline'em na tych konkretnych obejsciach.

Uruchom notebook trzy razy -- `MODEL_SIZE = "small"` (T4, darmowe),
`"big"` (A100, platne) i `"gemini"` (API, zewnetrzny hosting) -- i
porownaj `notebooks/llm_results_{small,big,gemini}.csv`. To jest
odpowiedz na otwarte pytanie #4 specyfikacji w calosci: nie tylko
"lokalnie czy zewnetrznie", ale i "czy w ogole oplaca sie placic za
wiekszy/zewnetrzny model, czy maly lokalny wystarczy". Jesli `"gemini"`
wygrywa wyraznie na ewazjach bez wiekszego FP rate, to mocny argument za
hostingiem zewnetrznym w Fazie 3 -- ale wtedy trzeba domkniecie: payloady
(nawet zredagowane) wysylane do cudzego API to realny temat prywatnosci,
patrz SPEC.md §11 i otwarte pytanie #4 o wycieku danych zadan do trzeciej
strony.